# 06 — Target Feasibility: y_addition_180d

**Objetivo:** medir si el primer target puede construirse sin confundir censura, ambigüedad temporal ni eventos previos a t0.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Target candidato
`y_addition_180d = 1` si existe una adición válida estrictamente posterior a `t0` y dentro de 180 días.

Un contrato solo puede ser negativo si el horizonte completo está observado.
En esta fase el objetivo es **GO / REDEFINE**, no entrenar un modelo.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.targets.addition import analyze_addition_target_feasibility, addition_target_by_year
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]
target_cfg = PROJECT_CONFIG["target"]

horizon_summary, target_diagnostics = analyze_addition_target_feasibility(
    FILES["contracts"],
    FILES["additions"],
    contract_id_column=target_cfg["contract_id_column"],
    t0_column=target_cfg["candidate_t0_column"],
    addition_contract_id_column=target_cfg["addition_contract_id_column"],
    addition_event_column=target_cfg["addition_event_column"],
    horizons=(30, 90, 180, 365),
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    horizon_summary,
    paths.reports / "profiling" / "target_feasibility_horizons.parquet",
)
write_report(
    target_diagnostics,
    paths.reports / "profiling" / "target_feasibility_diagnostics.parquet",
)

horizon_summary

In [ ]:
target_diagnostics.T

In [ ]:
by_year = addition_target_by_year(
    FILES["contracts"],
    FILES["additions"],
    horizon_days=target_cfg["horizon_days"],
    contract_id_column=target_cfg["contract_id_column"],
    t0_column=target_cfg["candidate_t0_column"],
    addition_contract_id_column=target_cfg["addition_contract_id_column"],
    addition_event_column=target_cfg["addition_event_column"],
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    by_year,
    paths.reports / "profiling" / "target_feasibility_by_year.parquet",
)

by_year

## Decisión obligatoria al finalizar
Documenta en `docs/target-specs/y_addition_180d.md`:
1. `t0` aprobado o rechazado.
2. timestamp del evento.
3. cutoff de etiquetas.
4. población excluida y por qué.
5. prevalencia global y por año.
6. cantidad de observaciones censuradas.
7. definición exacta de qué tipos de adición cuentan.
8. decisión **GO / REDEFINE TARGET**.

Si estas condiciones no están resueltas, no pases a Silver/feature engineering por inercia.